In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 77.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving trial_data_multiple_choice.tsv to trial_data_multiple_choice.tsv


In [ ]:
INPUT_PATH = "/content/trial_data_multiple_choice.tsv"

In [ ]:
import pandas as pd, csv

df = pd.read_csv(
    INPUT_PATH,
    sep="\t",
    quotechar='"',
    quoting=csv.QUOTE_MINIMAL,
    engine="python",
)

df["options"] = df["multiple_choice_options"].astype(str).str.split("\n")

print("Rows:", len(df))
print("Columns:", list(df.columns))

lens = df["options"].apply(len)
print("Options length distribution:\n", lens.value_counts())

display(df.head(3)[["index", "lang_reg", "question", "options"] + (["correct_answer"] if "correct_answer" in df.columns else [])])


Rows: 148
Columns: ['index', 'lang_reg', 'question', 'multiple_choice_options', 'correct_answer', 'options']
Options length distribution:
 options
4    146
3      2
Name: count, dtype: int64


,index,lang_reg,question,options,correct_answer
0,1,ms-SG,Apakah akronim lazim untuk flat perumahan awam...,"[DBS, HPB, HDB, SAF]",HDB
1,2,ms-SG,Parti politik manakah yang telah menjadi parti...,"[Parti Pekerja (WP) , Parti Tindakan Rakyat (P...",Parti Tindakan Rakyat (PAP)
2,3,ms-SG,"Apakah maskot rasmi Singapura, iaitu makhluk m...","[Merlion , Singa Kesopanan (Singa The Courtesy...",Merlion


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("GPU available:", torch.cuda.is_available())
print("Device map:", model.hf_device_map)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

GPU available: True
Device map: {'': 0}


In [ ]:
import torch
import torch.nn.functional as F

def build_prompt(lang_reg: str, question: str, options: list[str]) -> str:
    labels = [chr(ord("A")+i) for i in range(len(options))]
    opt_block = "\n".join([f"{labels[i]}. {options[i]}" for i in range(len(options))])
    return (
        f"You are answering from the perspective of someone living in: {lang_reg}.\n"
        "Choose the option that best matches everyday cultural norms and what is most typical there.\n"
        "Return the exact option text.\n\n"
        f"Question: {question}\n"
        f"Options:\n{opt_block}\n\n"
        "Answer: "
    )

@torch.no_grad()
def score_options(prompt_text: str, options: list[str]) -> list[float]:
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    P = len(prompt_ids)

    seqs, opt_lens = [], []
    for opt in options:
        opt_ids = tokenizer(opt, add_special_tokens=False).input_ids
        opt_lens.append(len(opt_ids))
        seqs.append(prompt_ids + opt_ids)

    max_len = max(len(s) for s in seqs)
    pad_id = tokenizer.pad_token_id

    input_ids = torch.full((len(seqs), max_len), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((len(seqs), max_len), dtype=torch.long)
    for i, s in enumerate(seqs):
        input_ids[i, :len(s)] = torch.tensor(s, dtype=torch.long)
        attention_mask[i, :len(s)] = 1

    device = next(model.parameters()).device
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]
    log_probs = F.log_softmax(shift_logits, dim=-1)

    scores = []
    for i in range(len(options)):
        O = opt_lens[i]
        if O == 0:
            scores.append(float("-inf"))
            continue

        start = max(P - 1, 0)
        end = start + O

        lp = log_probs[i, start:end, :]
        tgt = shift_labels[i, start:end]
        token_lps = lp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
        scores.append(token_lps.mean().item())

    return scores

def predict_row(row):
    opts = [o.strip() for o in row["options"] if str(o).strip() != ""]
    if len(opts) < 2:
        return opts[0] if len(opts) == 1 else ""

    prompt = build_prompt(str(row["lang_reg"]), str(row["question"]), opts)
    scores = score_options(prompt, opts)
    best = int(max(range(len(scores)), key=lambda i: scores[i]))
    return opts[best]


In [ ]:
from tqdm import tqdm

preds = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    preds.append(predict_row(row))

df["prediction"] = preds


100%|██████████| 148/148 [00:18<00:00,  7.93it/s]


In [ ]:
from tqdm import tqdm
import numpy as np

preds = []
chosen_idx = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    options = row["options"]
    options = [o.strip() for o in options if str(o).strip() != ""]

    prompt = build_prompt(str(row["lang_reg"]), str(row["question"]), options)

    scores = score_options(prompt, options)
    j = int(np.argmax(scores))

    preds.append(options[j])
    chosen_idx.append(j)

df["prediction"] = preds
df["pred_choice_index"] = chosen_idx

display(df.head(5)[["index", "lang_reg", "question", "options", "prediction"] + (["correct_answer"] if "correct_answer" in df.columns else [])])

100%|██████████| 148/148 [00:31<00:00,  4.73it/s]


,index,lang_reg,question,options,prediction,correct_answer
0,1,ms-SG,Apakah akronim lazim untuk flat perumahan awam...,"[DBS, HPB, HDB, SAF]",HPB,HDB
1,2,ms-SG,Parti politik manakah yang telah menjadi parti...,"[Parti Pekerja (WP) , Parti Tindakan Rakyat (P...",Parti Tindakan Rakyat (PAP),Parti Tindakan Rakyat (PAP)
2,3,ms-SG,"Apakah maskot rasmi Singapura, iaitu makhluk m...","[Merlion , Singa Kesopanan (Singa The Courtesy...",Singa Kesopanan (Singa The Courtesy Lion),Merlion
3,4,ms-SG,Apakah nama lapangan terbang antarabangsa utam...,"[Lapangan Terbang Seletar , Lapangan Terbang C...",Lapangan Terbang Seletar,Lapangan Terbang Changi
4,5,ms-SG,Singapura menyambut Hari Kebangsaan setiap tah...,"[Februari , Julai , September , Ogos]",Februari,Ogos


In [ ]:
lens = df["options"].apply(lambda x: len([o for o in x if str(o).strip() != ""]))
assert (lens >= 2).all(), "Some rows have <2 options"
print("✅ Options check passed (>=2).")

def is_valid(row):
    opts = [o.strip() for o in row["options"] if str(o).strip()!=""]
    return str(row["prediction"]).strip() in opts

assert df.apply(is_valid, axis=1).all(), "Some predictions are not among the options"
print("✅ Prediction-in-options check passed.")

if "correct_answer" in df.columns:
    acc = (df["prediction"] == df["correct_answer"]).mean()
    print("Pilot accuracy:", acc)


✅ Options check passed (>=2).
✅ Prediction-in-options check passed.
Pilot accuracy: 0.2972972972972973


In [ ]:
if "correct_answer" in df.columns:
    acc = (df["prediction"] == df["correct_answer"]).mean()
    print("Accuracy (trial/pilot):", acc)

    by_reg = df.groupby("lang_reg").apply(lambda g: (g["prediction"] == g["correct_answer"]).mean()).sort_values(ascending=False)
    display(by_reg.head(20))

Accuracy (trial/pilot): 0.2972972972972973


/tmp/ipython-input-3320669153.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_reg = df.groupby("lang_reg").apply(lambda g: (g["prediction"] == g["correct_answer"]).mean()).sort_values(ascending=False)


,0
lang_reg,
en-AU,0.714286
id-ID,0.600000
bg-BG,0.571429
ja-JP,0.571429
es-EC,0.500000
ta-LK,0.428571
ar-EG,0.428571
el-GR,0.400000
tl-PH,0.375000


In [ ]:
OUT_SUBMISSION = "/content/submission.tsv"
sub = df[["index","prediction"]].copy()

assert sub["index"].is_unique, "Duplicate index"
assert sub["prediction"].notna().all(), "Missing prediction"
assert len(sub) == len(df), "Row count mismatch"

sub.to_csv(OUT_SUBMISSION, sep="\t", index=False)
print("✅ Wrote:", OUT_SUBMISSION)

✅ Wrote: /content/submission.tsv


Verification

In [ ]:
import pandas as pd

OUT_SUBMISSION = "/content/submission.tsv"
sub = pd.read_csv(OUT_SUBMISSION, sep="\t")

print("Shape:", sub.shape)
print("Columns:", list(sub.columns))
display(sub.head(10))

Shape: (148, 2)
Columns: ['index', 'prediction']


,index,prediction
0,1,HPB
1,2,Parti Tindakan Rakyat (PAP)
2,3,Singa Kesopanan (Singa The Courtesy Lion)
3,4,Lapangan Terbang Seletar
4,5,Februari
5,6,Marina Bay Sands
6,7,Rupiah
7,8,DBS
8,9,சிங்கப்பூர் முன்னேற்றக் கட்சி (PSP)
9,10,சிங்கா எனும் உபசரிப்பு சிங்கம் (Singa the Cour...


In [ ]:
assert list(sub.columns) == ["index", "prediction"], "Wrong columns or order"

assert len(sub) == len(df), "Row count mismatch"

assert sub["index"].is_unique, "Duplicate index values"
assert sub["prediction"].notna().all(), "Missing predictions"

print("✅ Basic submission format checks passed.")

✅ Basic submission format checks passed.


In [ ]:
opt_map = df.set_index("index")["options"].to_dict()

def pred_valid(i, pred):
    opts = [o.strip() for o in opt_map[i] if str(o).strip() != ""]
    return str(pred).strip() in opts

bad = []
for i, pred in zip(sub["index"], sub["prediction"]):
    if i not in opt_map or not pred_valid(i, pred):
        bad.append(i)

print("Invalid rows:", len(bad))
if bad:
    print("First few invalid indices:", bad[:10])

Invalid rows: 0


In [ ]:
import random

for i in random.sample(list(sub["index"]), 5):
    row = df[df["index"] == i].iloc[0]
    print("\nINDEX:", i, "| lang_reg:", row["lang_reg"])
    print("Q:", row["question"])
    print("Options:", row["options"])
    pred = sub[sub["index"] == i]["prediction"].iloc[0]
    print("Prediction:", pred)



INDEX: 5 | lang_reg: ms-SG
Q: Singapura menyambut Hari Kebangsaan setiap tahun pada bulan apa?
Options: ['Februari ', 'Julai ', 'September ', 'Ogos']
Prediction: Februari

INDEX: 7 | lang_reg: ms-SG
Q: Apakah mata wang rasmi Singapura?
Options: ['Ringgit ', 'Baht ', 'Dolar ', 'Rupiah']
Prediction: Rupiah

INDEX: 139 | lang_reg: bg-BG
Q: Какво празнуваме на 24 май?
Options: ['Великден', 'Ден на независимостта', 'Ден на труда', 'Ден на българската просвета и култура и на славянската писменост']
Prediction: Ден на българската просвета и култура и на славянската писменост

INDEX: 42 | lang_reg: es-ES
Q: ¿Cuál es la obra literaria española más reconocida internacionalmente?
Options: ['El cantar del Mío Cid', 'Don Quijote de la Mancha', 'Las aventuras de Huckleberry Finn', 'Los tres mosqueteros']
Prediction: El cantar del Mío Cid

INDEX: 130 | lang_reg: tl-PH
Q: Sino ang tinaguriang pambansang bayani ng Pilipinas?
Options: ['Andres Bonifacio', 'Emilio Aguinaldo', 'Apolinario Mabini', 'Jose 

In [ ]:
if "correct_answer" in df.columns:
    merged = df.merge(sub, on="index", how="inner", suffixes=("", "_sub"))
    acc = (merged["prediction_sub"] == merged["correct_answer"]).mean()
    print("Accuracy vs gold (trial/pilot):", acc)


Accuracy vs gold (trial/pilot): 0.2972972972972973


In [ ]:
merged = df.merge(sub, on="index", how="inner", suffixes=("", "_sub"))
merged["correct"] = (merged["prediction_sub"] == merged["correct_answer"])

overall_acc = merged["correct"].mean()
per_lang = merged.groupby("lang_reg")["correct"].mean().sort_values(ascending=False)
macro_acc = per_lang.mean()

print("Overall accuracy:", overall_acc)
print("Macro (avg over lang_reg):", macro_acc)
print(per_lang)


Overall accuracy: 0.2972972972972973
Macro (avg over lang_reg): 0.29192546583850937
lang_reg
en-AU    0.714286
id-ID    0.600000
bg-BG    0.571429
ja-JP    0.571429
es-EC    0.500000
ta-LK    0.428571
ar-EG    0.428571
el-GR    0.400000
tl-PH    0.375000
ga-IE    0.285714
en-GB    0.200000
zh-CN    0.200000
ko-KR    0.200000
fa-IR    0.200000
es-ES    0.200000
ta-SG    0.142857
ar-MA    0.142857
eu-ES    0.142857
ms-SG    0.142857
zh-SG    0.142857
fr-FR    0.125000
ar-SA    0.000000
es-MX    0.000000
Name: correct, dtype: float64
